In [1]:
!pip install -q unsloth
!pip install -q trl datasets transformers accelerate peft bitsandbytes huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.1/81.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.4/24.4 MB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 110.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import os
import json
import torch

PROJECT_DIR = "/content/drive/MyDrive/novacart-domain-ai-assistant"

DATA_DIR = f"{PROJECT_DIR}/data"
MODEL_DIR = f"{PROJECT_DIR}/models"
REPORT_DIR = f"{PROJECT_DIR}/reports"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

print(PROJECT_DIR)

/content/drive/MyDrive/novacart-domain-ai-assistant


In [4]:
import os
from getpass import getpass
from huggingface_hub import login, whoami

os.environ["HF_TOKEN"] = getpass("Enter Hugging Face WRITE token: ")

login(token=os.environ["HF_TOKEN"])

print(whoami())

Enter Hugging Face WRITE token: ··········


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


{'type': 'user', 'id': '65b34aaf2b3c9da0b4f893f1', 'name': 'Devu758', 'fullname': 'Devanshu Shrivastava', 'email': 'devu7589@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1790812800, 'isPro': False, 'avatarUrl': '/avatars/898353eadfa2fa6723215952f292fb4e.svg', 'orgs': [{'type': 'org', 'id': '636fad178305bc19758fea13', 'name': 'IITMandi', 'fullname': 'Indian Institute of Technology Mandi', 'email': None, 'canPay': False, 'billingMode': 'postpaid', 'periodEnd': None, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/noauth/_RAM4U3_h6b7bQUcqqJGR.png', 'roleInOrg': 'write'}], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'LLM_Course_write', 'role': 'write', 'createdAt': '2026-09-26T13:45:22.834Z'}}}


In [5]:
HF_USERNAME = "Devu758"

STAGE1_REPO = f"{HF_USERNAME}/novacart-qwen-stage1-domain"
SFT_REPO = f"{HF_USERNAME}/novacart-qwen-sft"

print("Stage 1:", STAGE1_REPO)
print("Stage 2:", SFT_REPO)

Stage 1: Devu758/novacart-qwen-stage1-domain
Stage 2: Devu758/novacart-qwen-sft


In [6]:
evaluation_questions = [
    "How can I cancel my NovaCart order?",
    "How many days do I have to return a product?",
    "How long does NovaCart take to process a refund?",
    "What should I do if my product arrives damaged?",
    "How can I track my order?",
    "How long does standard delivery usually take?",
    "What should I do if my payment fails?",
    "What should I do if I was charged twice?",
    "Can I change my delivery address after placing an order?",
    "What should I do if an item is missing from my package?"
]

len(evaluation_questions)

10

In [7]:
from unsloth import FastLanguageModel

max_seq_length = 1024

base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.11: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [8]:
FastLanguageModel.for_inference(base_model)

def generate_answer(model, tokenizer, question, max_new_tokens=120):

    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )

    generated_tokens = outputs[0][inputs.shape[-1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return answer.strip()

In [9]:
base_results = []

for i, question in enumerate(evaluation_questions, start=1):

    answer = generate_answer(
        base_model,
        base_tokenizer,
        question
    )

    base_results.append({
        "question": question,
        "base_answer": answer
    })

    print("=" * 80)
    print(f"QUESTION {i}")
    print(question)
    print("\nBASE ANSWER:")
    print(answer)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION 1
How can I cancel my NovaCart order?

BASE ANSWER:
I'm sorry, but as an AI language model, I am not able to provide you with specific information about how to cancel your NovaCart order. However, generally speaking, the process of cancelling an order on any online shopping platform may vary depending on the website and its policies. Here are some general steps that you might follow:

1. Log in to your account: First, log into your NovaCart account.

2. Review your order history: Check your order history to see if there is an option to cancel or refund your order.

3. Contact customer support: If you cannot find an option


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION 2
How many days do I have to return a product?

BASE ANSWER:
The specific number of days for returns can vary depending on the store or company you're purchasing from. It's important to check with the seller directly as their policies may differ.

However, in general:

1. **Amazon**: Amazon typically allows 30 days for returns.
2. **Apple Store**: Apple offers a 14-day return policy.
3. **Best Buy**: Best Buy has a 30-day return period.
4. **Walmart**: Walmart usually provides a 30-day return window.
5. **Target**: Target generally offers a 30-day return option


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

QUESTION 7
What should I do if my payment fails?

BASE ANSWER:
If your payment fails, there are several steps you can take to resolve the issue:

1. Check your internet connection: Ensure that your internet is working properly and try again.

2. Verify your credit card information: Double-check that all details (card number, expiration date, CVV) are correct.

3. Contact your bank or credit card issuer: Reach out to them for assistance. They may be able to help identify the problem.

4. Try using a different payment method: If possible, switch to another payment option like PayPal or Apple Pay.

5. Wait 24-48


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION 8
What should I do if I was charged twice?

BASE ANSWER:
If you were charged twice for something that you did not actually do, it is important to address this issue as soon as possible. Here are some steps you can take:

1. **Review the Charges**: First, review the charges in question to understand what they represent and why they might be incorrect.

2. **Contact Your Credit Card Company or Bank**: Reach out to your credit card company or bank to explain the situation. They may have an internal process to correct these errors.

3. **File a Dispute with Your Credit Bureau**: If the charge appears on your credit report, you can dispute


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION 9
Can I change my delivery address after placing an order?

BASE ANSWER:
Yes, you can usually change your delivery address after placing an order. However, the specific process may vary depending on the online store or platform you're using. Here are some general steps to follow:

1. Log in to your account: Access your account and navigate to the "My Orders" or "Order History" section.

2. Review your orders: Check if there is any option to update your shipping information for the order(s) you placed.

3. Update your shipping details: If available, enter your new shipping address and click "Update Address."

4. Confirm changes: Verify that the
QUESTION 10
What should I do if an item is missing from my package?

BASE ANSWER:
If you have received an item and it's missing, here are some steps you can take:

1. Check the tracking information: Make sure that your order was shipped to the correct address and that there were no errors in the shipping process.

2. Contact the seller o

In [10]:
base_results_path = f"{REPORT_DIR}/base_model_results.json"

with open(base_results_path, "w", encoding="utf-8") as f:
    json.dump(base_results, f, indent=4, ensure_ascii=False)

print("Saved:", base_results_path)

Saved: /content/drive/MyDrive/novacart-domain-ai-assistant/reports/base_model_results.json


In [11]:
md = """# Base Model Evaluation

Base Model: Qwen2.5-1.5B-Instruct

The original base model was evaluated on 10 NovaCart customer-support questions before instruction fine-tuning.

| # | Question | Base Model Answer | Problem |
|---|---|---|---|
"""

for i, result in enumerate(base_results, start=1):

    q = result["question"].replace("|", "\\|")
    a = result["base_answer"].replace("|", "\\|").replace("\n", " ")

    md += f"| {i} | {q} | {a} | Generic / not guaranteed to follow NovaCart-specific policy |\\n"

base_md_path = f"{REPORT_DIR}/base_model_evaluation.md"

with open(base_md_path, "w", encoding="utf-8") as f:
    f.write(md)

print("Saved:", base_md_path)

Saved: /content/drive/MyDrive/novacart-domain-ai-assistant/reports/base_model_evaluation.md


In [12]:
del base_model
del base_tokenizer

import gc

gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared.")

GPU memory cleared.


In [14]:
policies = {
    "cancellation":
        "NovaCart orders can be cancelled before shipment through the Orders section. "
        "Once the order has shipped, cancellation is unavailable and the customer may request a return after delivery.",

    "return":
        "Eligible NovaCart products may be returned within 30 days of delivery. "
        "Returns can be initiated from the Orders section.",

    "refund":
        "Approved NovaCart refunds are generally processed within 5 to 7 business days "
        "and returned to the original payment method.",

    "damaged":
        "A damaged product should be reported within 48 hours of delivery. "
        "The customer should provide the order number and photographs of the damage.",

    "wrong_product":
        "Customers who receive the wrong product should report it through the Help Center "
        "with the order number and product details. NovaCart may arrange a replacement or return.",

    "tracking":
        "Once an order ships, customers can track it from the Orders section using the Track Order option.",

    "delivery":
        "Standard NovaCart delivery generally takes 3 to 7 business days, "
        "depending on location, availability, courier capacity, and local conditions.",

    "payment_failure":
        "If a payment fails, customers should verify their payment details and balance "
        "and retry the payment or use another supported payment method.",

    "duplicate_charge":
        "Customers who see a duplicate charge should contact NovaCart support "
        "with their order number and transaction ID for investigation.",

    "missing_item":
        "Missing items should be reported within 48 hours of delivery with the order number "
        "and package details.",

    "warranty":
        "Warranty coverage depends on the individual product and manufacturer. "
        "Customers should check the warranty information on the product page.",

    "address":
        "Customers may change their delivery address before the order ships. "
        "After shipment begins, the address generally cannot be changed.",

    "coupon":
        "NovaCart coupons must normally be applied during checkout and usually cannot "
        "be applied after an order has already been placed.",

    "password":
        "Customers can reset their NovaCart password using the Forgot Password option "
        "on the sign-in page.",

    "support":
        "Customers can contact NovaCart through the Help Center and should provide relevant "
        "details such as their order number, transaction ID, product information, and issue description."
}

In [15]:
question_templates = {

    "cancellation": [
        "How can I cancel my NovaCart order?",
        "Can I cancel an order after placing it?",
        "Where can I cancel my order?",
        "Can I cancel an order that has already shipped?",
        "I changed my mind about my purchase. What should I do?",
        "What is NovaCart's cancellation policy?",
        "When is order cancellation allowed?",
        "My order has not shipped yet. Can I cancel it?"
    ],

    "return": [
        "How can I return a NovaCart product?",
        "What is NovaCart's return period?",
        "How many days do I have to return an item?",
        "Where do I start a return?",
        "Can I return an eligible product after delivery?",
        "What is the return policy?",
        "I don't want my delivered product. What can I do?",
        "How long after delivery can I request a return?"
    ],

    "refund": [
        "How long does NovaCart take to process a refund?",
        "When will I receive my refund?",
        "How many business days does a refund take?",
        "Where will my refund be sent?",
        "What is NovaCart's refund processing time?",
        "My return was approved. When should I expect my money?",
        "Does NovaCart refund to the original payment method?",
        "How are approved refunds processed?"
    ],

    "damaged": [
        "What should I do if my product arrives damaged?",
        "How quickly should I report a damaged product?",
        "My item arrived broken. What should I do?",
        "What information is required for a damaged-item claim?",
        "Do I need photographs of a damaged product?",
        "My package contained a damaged product. How do I report it?",
        "What is NovaCart's damaged-product policy?",
        "Can I report a damaged product two days after delivery?"
    ],

    "wrong_product": [
        "What should I do if NovaCart sends me the wrong item?",
        "I received a different product from what I ordered.",
        "How do I report an incorrect product?",
        "Can NovaCart replace a wrongly delivered product?",
        "Where should I report a wrong item?",
        "What details should I provide if I receive the wrong product?",
        "I ordered one product but received another. What now?",
        "What is the process for a wrongly delivered item?"
    ],

    "tracking": [
        "How can I track my NovaCart order?",
        "Where can I find order tracking?",
        "When does tracking information become available?",
        "Can I track my package after shipment?",
        "Where is the Track Order option?",
        "How do I check the status of a shipped order?",
        "I want to see where my NovaCart package is.",
        "Does NovaCart provide shipment tracking?"
    ],

    "delivery": [
        "How long does standard NovaCart delivery take?",
        "What is NovaCart's normal delivery time?",
        "How many days should standard shipping take?",
        "When should I expect my NovaCart package?",
        "Does delivery time vary by location?",
        "What can affect NovaCart delivery time?",
        "How quickly are standard orders delivered?",
        "What is the usual shipping duration?"
    ],

    "payment_failure": [
        "What should I do if my payment fails?",
        "My payment was declined during checkout. What should I do?",
        "Can I retry a failed NovaCart payment?",
        "Should I try another payment method if payment fails?",
        "Why should I check my balance after a failed payment?",
        "What steps should I take after a checkout payment failure?",
        "My transaction didn't complete. What should I do?",
        "How does NovaCart handle failed payments?"
    ],

    "duplicate_charge": [
        "What should I do if I was charged twice?",
        "NovaCart charged me twice for one order.",
        "How can I report a duplicate payment?",
        "What information should I provide for a duplicate charge?",
        "Who should I contact about duplicate billing?",
        "I see two transactions for one NovaCart order. What now?",
        "How does NovaCart investigate duplicate charges?",
        "Do I need my transaction ID for a duplicate-payment complaint?"
    ],

    "missing_item": [
        "What should I do if an item is missing from my package?",
        "How soon should I report a missing item?",
        "My order arrived incomplete. What should I do?",
        "What details are needed when reporting a missing item?",
        "One product is missing from my delivery.",
        "How do I report an incomplete NovaCart package?",
        "What is NovaCart's missing-item policy?",
        "Can I report a missing product within 48 hours?"
    ],

    "warranty": [
        "Does NovaCart provide product warranties?",
        "How can I check the warranty on a product?",
        "Is the warranty the same for every NovaCart product?",
        "Where can I find warranty information?",
        "Who determines the warranty coverage?",
        "Does warranty duration depend on the product?",
        "How do I know whether my item has a warranty?",
        "What is NovaCart's warranty policy?"
    ],

    "address": [
        "Can I change my delivery address?",
        "Can I update my address before shipment?",
        "Can my address be changed after the order ships?",
        "I entered the wrong shipping address. What should I do?",
        "When can a NovaCart delivery address be modified?",
        "My order has not shipped yet. Can I update the address?",
        "Why can't I change my address after shipment?",
        "What is NovaCart's address-change policy?"
    ],

    "coupon": [
        "How do I use a NovaCart coupon?",
        "Can I apply a coupon after placing my order?",
        "When should a promotional code be entered?",
        "Why isn't my coupon applying?",
        "Are coupons entered during checkout?",
        "Can a discount code be added after payment?",
        "Do NovaCart coupons have eligibility conditions?",
        "What is NovaCart's coupon policy?"
    ],

    "password": [
        "I forgot my NovaCart password. What should I do?",
        "How can I reset my password?",
        "Where is the Forgot Password option?",
        "I cannot sign in because I forgot my password.",
        "How do I recover access to my NovaCart account?",
        "Can NovaCart send me a password reset?",
        "What's the process for changing a forgotten password?",
        "How can I regain access to my account?"
    ],

    "support": [
        "How can I contact NovaCart customer support?",
        "Where can I get help with my order?",
        "What information should I give customer support?",
        "Does NovaCart have a Help Center?",
        "How can I make support resolve my issue faster?",
        "Should I include my order number in a support request?",
        "Where should I report a NovaCart problem?",
        "What details should be included in a support request?"
    ]
}

In [16]:
instruction_examples = []

for topic, questions in question_templates.items():

    answer = policies[topic]

    for question in questions:

        instruction_examples.append({
            "instruction": question,
            "response": answer
        })

print("Total examples:", len(instruction_examples))

Total examples: 120


In [17]:
for example in instruction_examples[:5]:
    print("QUESTION:", example["instruction"])
    print("ANSWER:", example["response"])
    print("-" * 80)

QUESTION: How can I cancel my NovaCart order?
ANSWER: NovaCart orders can be cancelled before shipment through the Orders section. Once the order has shipped, cancellation is unavailable and the customer may request a return after delivery.
--------------------------------------------------------------------------------
QUESTION: Can I cancel an order after placing it?
ANSWER: NovaCart orders can be cancelled before shipment through the Orders section. Once the order has shipped, cancellation is unavailable and the customer may request a return after delivery.
--------------------------------------------------------------------------------
QUESTION: Where can I cancel my order?
ANSWER: NovaCart orders can be cancelled before shipment through the Orders section. Once the order has shipped, cancellation is unavailable and the customer may request a return after delivery.
--------------------------------------------------------------------------------
QUESTION: Can I cancel an order that 

In [18]:
instruction_dataset_path = f"{DATA_DIR}/instruction_dataset.jsonl"

with open(
    instruction_dataset_path,
    "w",
    encoding="utf-8"
) as f:

    for example in instruction_examples:
        f.write(
            json.dumps(
                example,
                ensure_ascii=False
            ) + "\n"
        )

print("Saved:", instruction_dataset_path)

Saved: /content/drive/MyDrive/novacart-domain-ai-assistant/data/instruction_dataset.jsonl


In [19]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = STAGE1_REPO,
    max_seq_length = 1024,
    dtype = None,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.9.11: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.9.11 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [20]:
from datasets import Dataset

instruction_dataset = Dataset.from_list(instruction_examples)

def format_example(example):

    messages = [
        {
            "role": "user",
            "content": example["instruction"]
        },
        {
            "role": "assistant",
            "content": example["response"]
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    return {
        "text": text
    }


formatted_dataset = instruction_dataset.map(format_example)

print(formatted_dataset)

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

Dataset({
    features: ['instruction', 'response', 'text'],
    num_rows: 120
})


In [21]:
print(formatted_dataset[0]["text"])

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
How can I cancel my NovaCart order?<|im_end|>
<|im_start|>assistant
NovaCart orders can be cancelled before shipment through the Orders section. Once the order has shipped, cancellation is unavailable and the customer may request a return after delivery.<|im_end|>



In [22]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

sft_config = SFTConfig(

    output_dir = "outputs_sft",

    dataset_text_field = "text",
    max_length = 1024,

    per_device_train_batch_size = 2,

    gradient_accumulation_steps = 4,

    num_train_epochs = 2,

    learning_rate = 2e-4,

    warmup_ratio = 0.1,

    weight_decay = 0.01,

    fp16 = not is_bfloat16_supported(),

    bf16 = is_bfloat16_supported(),

    optim = "adamw_8bit",

    logging_steps = 1,

    seed = 42,

    report_to = "none",
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [23]:
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = formatted_dataset,
    args = sft_config,
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/120 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [24]:
sft_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 120 | Num Epochs = 2 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
/usr/local/lib/python3.13/dist-packages/unsloth/import_fixes.py:4091: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_avail

Step,Training Loss
1,3.645559
2,3.653433
3,2.829686
4,2.388976
5,1.943343
6,1.520528
7,1.319243
8,1.108173
9,1.012087
10,0.858658


Unsloth: Restored added_tokens_decoder metadata in outputs_sft/checkpoint-30/tokenizer_config.json.


In [25]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536, padding_idx=151654)
        (layers): ModuleList(
          (0): Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
       

In [26]:
question = "How long does NovaCart take to process a refund?"

answer = generate_answer(
    model,
    tokenizer,
    question
)

print("QUESTION:")
print(question)

print("\nSFT ANSWER:")
print(answer)

Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION:
How long does NovaCart take to process a refund?

SFT ANSWER:
Refunds generally begin processing within 5 to 7 business days after the transaction is complete and usually take 3 to 7 business days to arrive.


In [27]:
sft_results = []

for i, question in enumerate(
    evaluation_questions,
    start=1
):

    answer = generate_answer(
        model,
        tokenizer,
        question
    )

    sft_results.append({
        "question": question,
        "sft_answer": answer
    })

    print("=" * 80)
    print(f"QUESTION {i}")
    print(question)

    print("\nSFT ANSWER:")
    print(answer)

Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION 1
How can I cancel my NovaCart order?

SFT ANSWER:
Customers who receive the wrong product should report it through the Help Center with the order number and product details.


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION 2
How many days do I have to return a product?

SFT ANSWER:
Returns can be initiated within 30 days of delivery. Returns may not be available on certain products and may take longer for delivery depending on the availability of replacement parts.


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

QUESTION 7
What should I do if my payment fails?

SFT ANSWER:
Customers who receive an incorrect product should report it through the Help Center with the order number and product details.


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION 8
What should I do if I was charged twice?

SFT ANSWER:
Customers who see an unexpected charge can contact NovaCart support with their order number and transaction ID for investigation.


Both `max_new_tokens` (=120) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


QUESTION 9
Can I change my delivery address after placing an order?

SFT ANSWER:
Once the order ships, the customer may change their delivery address before shipment begins. After shipment has begun, the address generally cannot be changed.
QUESTION 10
What should I do if an item is missing from my package?

SFT ANSWER:
Missing items should be reported within 48 hours of delivery with the order number and package details.


In [28]:
sft_results_path = f"{REPORT_DIR}/sft_model_results.json"

with open(
    sft_results_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        sft_results,
        f,
        indent=4,
        ensure_ascii=False
    )

print("Saved:", sft_results_path)

Saved: /content/drive/MyDrive/novacart-domain-ai-assistant/reports/sft_model_results.json


In [29]:
sft_training_config = {

    "starting_model":
        "Devu758/novacart-qwen-stage1-domain",

    "training_stage":
        "Instruction Fine-Tuning / SFT",

    "dataset_size":
        len(formatted_dataset),

    "epochs":
        2,

    "per_device_batch_size":
        2,

    "gradient_accumulation_steps":
        4,

    "effective_batch_size":
        8,

    "learning_rate":
        2e-4,

    "max_seq_length":
        1024,

    "quantization":
        "4-bit",

    "lora_rank":
        16,

    "lora_alpha":
        16
}


with open(
    f"{REPORT_DIR}/stage2_sft_config.json",
    "w"
) as f:

    json.dump(
        sft_training_config,
        f,
        indent=4
    )

In [30]:
with open(
    f"{REPORT_DIR}/stage2_sft_logs.json",
    "w"
) as f:

    json.dump(
        trainer.state.log_history,
        f,
        indent=4
    )

print("SFT config and logs saved.")

SFT config and logs saved.


In [31]:
sft_drive_path = (
    f"{MODEL_DIR}/novacart-qwen-sft"
)

model.save_pretrained(
    sft_drive_path
)

tokenizer.save_pretrained(
    sft_drive_path
)

print(
    "SFT adapter saved:",
    sft_drive_path
)

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/novacart-domain-ai-assistant/models/novacart-qwen-sft/tokenizer_config.json.


SFT adapter saved: /content/drive/MyDrive/novacart-domain-ai-assistant/models/novacart-qwen-sft


In [32]:
from huggingface_hub import create_repo

create_repo(
    repo_id = SFT_REPO,
    repo_type = "model",
    private = False,
    exist_ok = True,
    token = os.environ["HF_TOKEN"],
)

RepoUrl('https://huggingface.co/Devu758/novacart-qwen-sft', endpoint='https://huggingface.co', repo_type='model', repo_id='Devu758/novacart-qwen-sft')

In [33]:
model.push_to_hub(
    SFT_REPO,
    token = os.environ["HF_TOKEN"]
)

tokenizer.push_to_hub(
    SFT_REPO,
    token = os.environ["HF_TOKEN"]
)

print(
    f"https://huggingface.co/{SFT_REPO}"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.7kB / 73.9MB            

Saved model to https://huggingface.co/Devu758/novacart-qwen-sft


README.md:   0%|          | 0.00/5.18k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpbyac8o8e/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpbyac8o8e/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

https://huggingface.co/Devu758/novacart-qwen-sft
